# Análise Empírica e Assintótica: Fibonacci Heap
**Disciplina:** Algoritmos Avançados — CEFET-MG / UFCI-MG  
**Autores:** Pedro Henrique de Araújo Carvalho e Igor Peixoto Rodrigues  

Este notebook realiza a leitura dos dados experimentais brutos (`.txt`), executa o processamento estatístico das **10 repetições** e gera os gráficos de desempenho e consumo de memória.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estilo padronizado para gráficos acadêmicos
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'lines.linewidth': 1.8,
    'lines.markersize': 6,
    'grid.alpha': 0.4
})

## 1. Carregamento dos Dados Brutos (.txt)
Os dados foram coletados para $N \in \{100, 1.000, 5.000, 10.000, 50.000, 100.000, 500.000, 1.000.000\}$ com 10 repetições para cada tamanho.

In [ ]:
def carregar_dados_tempo(caminho):
    dados = {}
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if not linha or linha.startswith('#'):
                continue
            partes = linha.split()
            if len(partes) < 5:
                continue
            est, op, n_str, rep_str, tempo_str = partes[0], partes[1], partes[2], partes[3], partes[4]
            n = int(n_str)
            tempo_us = float(tempo_str)
            chave = (est, op, n)
            if chave not in dados:
                dados[chave] = []
            dados[chave].append(tempo_us)
    return dados

def carregar_dados_memoria(caminho):
    dados = {}
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if not linha or linha.startswith('#'):
                continue
            partes = linha.split()
            if len(partes) < 5:
                continue
            est, n_str, rep_str, mem_teor_str, mem_proc_str = partes[0], partes[1], partes[2], partes[3], partes[4]
            n = int(n_str)
            mem_teor = float(mem_teor_str)
            mem_proc = float(mem_proc_str)
            chave = (est, n)
            if chave not in dados:
                dados[chave] = {'teorica': [], 'processo': []}
            dados[chave]['teorica'].append(mem_teor)
            dados[chave]['processo'].append(mem_proc)
    return dados

dados_tempo = carregar_dados_tempo('data/tempo_por_entrada.txt')
dados_mem = carregar_dados_memoria('data/memoria_por_entrada.txt')
tamanhos_n = sorted(list(set(k[2] for k in dados_tempo.keys())))
print(f"Tamanhos de entrada avaliados: {tamanhos_n}")

## 2. Gráfico 1: Tempo Total de Execução vs. Tamanho da Entrada
Compara o tempo total (em milissegundos) para inserção, extração do mínimo e diminuição de chave (*decrease-key*).

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=150)

y_fib_ins = [np.mean(dados_tempo.get(('fibonacci_heap', 'insert', n), [0])) / 1000.0 for n in tamanhos_n]
err_fib_ins = [np.std(dados_tempo.get(('fibonacci_heap', 'insert', n), [0])) / 1000.0 for n in tamanhos_n]
y_pq_ins = [np.mean(dados_tempo.get(('priority_queue', 'insert', n), [0])) / 1000.0 for n in tamanhos_n]
err_pq_ins = [np.std(dados_tempo.get(('priority_queue', 'insert', n), [0])) / 1000.0 for n in tamanhos_n]

y_fib_ext = [np.mean(dados_tempo.get(('fibonacci_heap', 'extract_min', n), [0])) / 1000.0 for n in tamanhos_n]
err_fib_ext = [np.std(dados_tempo.get(('fibonacci_heap', 'extract_min', n), [0])) / 1000.0 for n in tamanhos_n]
y_pq_ext = [np.mean(dados_tempo.get(('priority_queue', 'extract_min', n), [0])) / 1000.0 for n in tamanhos_n]
err_pq_ext = [np.std(dados_tempo.get(('priority_queue', 'extract_min', n), [0])) / 1000.0 for n in tamanhos_n]

y_fib_dec = [np.mean(dados_tempo.get(('fibonacci_heap', 'decrease_key', n), [0])) / 1000.0 for n in tamanhos_n]
err_fib_dec = [np.std(dados_tempo.get(('fibonacci_heap', 'decrease_key', n), [0])) / 1000.0 for n in tamanhos_n]

ax.errorbar(tamanhos_n, y_fib_ins, yerr=err_fib_ins, fmt='-o', color='#1f4e79', label='Fibonacci Heap: Inserção [O(1)]', capsize=3)
ax.errorbar(tamanhos_n, y_pq_ins, yerr=err_pq_ins, fmt='--s', color='#2e7d32', label='std::priority_queue: Inserção [O(log n)]', capsize=3)
ax.errorbar(tamanhos_n, y_fib_ext, yerr=err_fib_ext, fmt='-^', color='#c00000', label='Fibonacci Heap: Extração Mínimo [O(log n)]', capsize=3)
ax.errorbar(tamanhos_n, y_pq_ext, yerr=err_pq_ext, fmt='--v', color='#e36c09', label='std::priority_queue: Extração Mínimo [O(log n)]', capsize=3)
ax.errorbar(tamanhos_n, y_fib_dec, yerr=err_fib_dec, fmt='-d', color='#7030a0', label='Fibonacci Heap: Decrease-Key (N/10) [O(1)]', capsize=3)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho da Entrada (N elementos)')
ax.set_ylabel('Tempo Médio (milissegundos)')
ax.set_title('Tempo de Processamento vs. Tamanho da Entrada (10 Repetições)')
ax.grid(True, which="both", ls=":")
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 3. Gráfico 2: Consumo de Memória vs. Tamanho da Entrada
Compara o modelo analítico ($48N + 16$ bytes para Fibonacci Heap vs. $4N$ bytes para Heap Binário em vetor) e o consumo empírico do processo.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=150)

mem_fib_mb = [(n * 48.0 + 16.0) / (1024.0 * 1024.0) for n in tamanhos_n]
mem_pq_mb = [(n * 4.0) / (1024.0 * 1024.0) for n in tamanhos_n]

ax.plot(tamanhos_n, mem_fib_mb, '-o', color='#1f4e79', label='Fibonacci Heap: Modelo Teórico (48 B/nó)')
ax.plot(tamanhos_n, mem_pq_mb, '--s', color='#2e7d32', label='std::priority_queue: Modelo Teórico (4 B/chave)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho da Entrada (N elementos)')
ax.set_ylabel('Consumo de Memória (Megabytes - MB)')
ax.set_title('Consumo de Memória vs. Tamanho da Entrada')
ax.grid(True, which="both", ls=":")
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Gráfico 3: Custo Unitário por Operação (Tempo / Operação)
Demonstra a natureza assintótica unitária das operações: inserção e decrease-key permanecem com tempo unitário constante ($O(1)$), enquanto a extração cresce com $\log n$ devido à consolidação.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=150)

custo_unit_fib_ins = [(y_fib_ins[i] * 1000.0) / tamanhos_n[i] for i in range(len(tamanhos_n))]
custo_unit_pq_ins = [(y_pq_ins[i] * 1000.0) / tamanhos_n[i] for i in range(len(tamanhos_n))]
custo_unit_fib_ext = [(y_fib_ext[i] * 1000.0) / tamanhos_n[i] for i in range(len(tamanhos_n))]
custo_unit_pq_ext = [(y_pq_ext[i] * 1000.0) / tamanhos_n[i] for i in range(len(tamanhos_n))]
custo_unit_fib_dec = [(y_fib_dec[i] * 1000.0) / max(10, tamanhos_n[i] / 10) for i in range(len(tamanhos_n))]

ax.plot(tamanhos_n, custo_unit_fib_ins, '-o', color='#1f4e79', label='Fibonacci Heap: Inserção Unitária [O(1)]')
ax.plot(tamanhos_n, custo_unit_fib_dec, '-d', color='#7030a0', label='Fibonacci Heap: Decrease-Key Unitária [O(1)]')
ax.plot(tamanhos_n, custo_unit_pq_ins, '--s', color='#2e7d32', label='std::priority_queue: Inserção Unitária [O(log n)]')
ax.plot(tamanhos_n, custo_unit_fib_ext, '-^', color='#c00000', label='Fibonacci Heap: Extração Unitária [O(log n)]')
ax.plot(tamanhos_n, custo_unit_pq_ext, '--v', color='#e36c09', label='std::priority_queue: Extração Unitária [O(log n)]')

ax.set_xscale('log')
ax.set_xlabel('Tamanho da Entrada (N elementos)')
ax.set_ylabel('Custo Unitário Médio (µs por operação)')
ax.set_title('Custo Amortizado Unitário por Operação Individual')
ax.grid(True, which="both", ls=":")
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()